# Métricas del corpus shiwilu

Métricas descriptivas de `corpus_shiwilu_final.csv`: 700 pares bilingüe español–shiwilu
anotados con 7 categorías de intención comunicativa.

Tesis: *Evaluación de técnicas de aumento de datos y caracterización de embeddings
en la clasificación de intenciones para la lengua shiwilu*.

**Contenido:**
1. Carga y control de calidad
2. Distribución por categoría y fuente
3. Longitud de los enunciados
4. Vocabulario y riqueza léxica
5. Riqueza léxica por categoría
6. Tabla resumen

In [1]:
import re
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

# Raiz del repositorio: se busca hacia arriba el pyproject.toml, de modo que el
# notebook funcione sin importar desde donde se haya lanzado Jupyter.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from shiwilu.rutas import CORPUS_CSV, TABLAS

RUTA_CORPUS = CORPUS_CSV
RUTA_SALIDA = TABLAS / "metricas_corpus.csv"
TABLAS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 60)

## 1. Carga y control de calidad

Verificamos que el corpus esté completo: sin traducciones vacías, sin identificadores
duplicados y con las siete categorías presentes.

In [2]:
df = pd.read_csv(RUTA_CORPUS)

print(f"Total de pares          {len(df)}")
print(f"Columnas                {list(df.columns)}")
print(f"Categorías de intención {df['intencion'].nunique()}")
print(f"Valores nulos           {df.isna().sum().sum()}")
print(f"IDs duplicados          {df['id'].duplicated().sum()}")
print(f"Pares duplicados        {df.duplicated(subset=['espanol','shiwilu']).sum()}")

df.head()

Total de pares          700
Columnas                ['id', 'espanol', 'shiwilu', 'intencion', 'fuente']
Categorías de intención 7
Valores nulos           0
IDs duplicados          0
Pares duplicados        0


,id,espanol,shiwilu,intencion,fuente
0,AFI_NEW_001,Claro.,tekinchi,AFI,api_generada
1,AFI_NEW_002,Exacto.,nanapi'la,AFI,api_generada
2,AFI_NEW_003,Correcto.,Musu',AFI,api_generada
3,AFI_NEW_004,Verdad.,tekinchi,AFI,api_generada
4,AFI_NEW_005,Perfecto.,mupalli,AFI,api_generada


## 2. Distribución por categoría y fuente

Cada par proviene de las flashcards bilingüe existentes o de las oraciones generadas
con la API y traducidas posteriormente por el hablante nativo.

In [4]:
distribucion = (
    df.groupby(["intencion", "fuente"])
      .size()
      .unstack(fill_value=0)
      .rename(columns={"flashcards": "Flashcards", "api_generada": "API"})
)
distribucion["Total"] = distribucion.sum(axis=1)
distribucion.loc["TOTAL"] = distribucion.sum()
distribucion

fuente,API,Flashcards,Total
intencion,,,
AFI,91,9,100
DES,0,100,100
EMO,79,21,100
NEG,86,14,100
PRG,0,100,100
REQUEST,0,100,100
SAL,85,15,100
TOTAL,341,359,700


## 3. Longitud de los enunciados

Calculamos la longitud en palabras y en caracteres para ambos idiomas.
El ratio de compresión indica cuántas palabras del español corresponden a
una del shiwilu.

In [5]:
df["esp_pal"] = df["espanol"].str.split().str.len()
df["shi_pal"] = df["shiwilu"].str.split().str.len()
df["esp_car"] = df["espanol"].str.len()
df["shi_car"] = df["shiwilu"].str.len()

longitud = pd.DataFrame({
    "Español": [
        round(df["esp_pal"].mean(), 2),
        df["esp_pal"].max(),
        round(df["esp_car"].mean(), 2),
        round(df["esp_car"].sum() / df["esp_pal"].sum(), 2),
    ],
    "Shiwilu": [
        round(df["shi_pal"].mean(), 2),
        df["shi_pal"].max(),
        round(df["shi_car"].mean(), 2),
        round(df["shi_car"].sum() / df["shi_pal"].sum(), 2),
    ],
}, index=[
    "Palabras por enunciado (media)",
    "Palabras por enunciado (máximo)",
    "Caracteres por enunciado (media)",
    "Caracteres por palabra",
])

ratio = df["esp_pal"].sum() / df["shi_pal"].sum()
print(f"Ratio de compresión ES:SH (palabras) = {ratio:.2f}\n")
longitud

Ratio de compresión ES:SH (palabras) = 1.32



,Español,Shiwilu
Palabras por enunciado (media),2.18,1.66
Palabras por enunciado (máximo),5.00,3.00
Caracteres por enunciado (media),13.04,15.60
Caracteres por palabra,5.98,9.41


## 4. Vocabulario y riqueza léxica

Tokenizamos conservando el apóstrofe, que en shiwilu marca la oclusiva glotal
(Valenzuela & Gussenhoven, 2013). El Type-Token Ratio es la proporción de formas
distintas sobre el total de palabras.

In [ ]:
def tokenizar(texto):
    """Separa en palabras conservando el apóstrofe de la oclusiva glotal."""
    return re.findall(r"[A-Za-zÁÉÍÓÚÑÜáéíóúñü']+", texto.lower())


def metricas_lexicas(textos):
    """Devuelve (tokens, tipos, type-token ratio)."""
    tokens = [tok for texto in textos for tok in tokenizar(texto)]
    tipos = set(tokens)
    return len(tokens), len(tipos), len(tipos) / len(tokens)


tok_esp, tip_esp, ttr_esp = metricas_lexicas(df["espanol"])
tok_shi, tip_shi, ttr_shi = metricas_lexicas(df["shiwilu"])

vocabulario = pd.DataFrame({
    "Español": [tok_esp, tip_esp, round(ttr_esp, 3)],
    "Shiwilu": [tok_shi, tip_shi, round(ttr_shi, 3)],
}, index=["Tokens", "Tipos distintos", "Type-Token Ratio"])

vocabulario

## 5. Riqueza léxica por categoría

El Type-Token Ratio calculado dentro de cada categoría de intención permite
observar cuáles son léxicamente más variadas y cuáles más formulaicas.

In [ ]:
filas = []
for intencion in sorted(df["intencion"].unique()):
    sub = df[df["intencion"] == intencion]
    _, _, t_esp = metricas_lexicas(sub["espanol"])
    _, _, t_shi = metricas_lexicas(sub["shiwilu"])
    filas.append({
        "Intención": intencion,
        "TTR español": round(t_esp, 3),
        "TTR shiwilu": round(t_shi, 3),
    })

ttr_categoria = pd.DataFrame(filas).set_index("Intención")
ttr_categoria

## 6. Otros indicadores

Traducciones shiwilu que se repiten para distintos enunciados en español,
y presencia de la oclusiva glotal en el corpus.

In [ ]:
n_unicas = df["shiwilu"].nunique()
n_glotal = df["shiwilu"].str.contains("'").sum()

print(f"Traducciones shiwilu únicas      {n_unicas} de {len(df)}")
print(f"Enunciados con oclusiva glotal   {n_glotal} ({n_glotal / len(df):.1%})")

## 7. Tabla resumen

Consolidación de todas las métricas para citar en el informe.

In [ ]:
resumen = pd.DataFrame({
    "metrica": [
        "Total de pares",
        "Categorías de intención",
        "Pares por categoría",
        "Fuente: flashcards",
        "Fuente: api_generada",
        "Longitud media español (palabras)",
        "Longitud media shiwilu (palabras)",
        "Longitud media español (caracteres)",
        "Longitud media shiwilu (caracteres)",
        "Caracteres por palabra español",
        "Caracteres por palabra shiwilu",
        "Ratio de compresión ES:SH (palabras)",
        "Tokens español",
        "Tokens shiwilu",
        "Tipos español",
        "Tipos shiwilu",
        "TTR español",
        "TTR shiwilu",
        "Traducciones shiwilu únicas",
        "Enunciados con oclusiva glotal (%)",
    ],
    "valor": [
        len(df),
        df["intencion"].nunique(),
        int(df["intencion"].value_counts().mean()),
        int((df["fuente"] == "flashcards").sum()),
        int((df["fuente"] == "api_generada").sum()),
        round(df["esp_pal"].mean(), 2),
        round(df["shi_pal"].mean(), 2),
        round(df["esp_car"].mean(), 2),
        round(df["shi_car"].mean(), 2),
        round(df["esp_car"].sum() / df["esp_pal"].sum(), 2),
        round(df["shi_car"].sum() / df["shi_pal"].sum(), 2),
        round(ratio, 2),
        tok_esp,
        tok_shi,
        tip_esp,
        tip_shi,
        round(ttr_esp, 3),
        round(ttr_shi, 3),
        n_unicas,
        f"{n_glotal / len(df):.1%}",
    ],
})

resumen.to_csv(RUTA_SALIDA, index=False, encoding="utf-8")
print(f"Guardado en: {RUTA_SALIDA}\n")
resumen